In [25]:
import pandas as pd
from pathlib import Path

# Diretório principal do projeto
PROJECT_ROOT = Path.cwd().parent

# Origem: arquivos Parquet da camada Silver
SOURCE_DIR = (
    PROJECT_ROOT
    / "data"
    / "silver"
    / "open_meteo"
    / "forecast"
)

# Destino: arquivos Parquet da camada Gold
TARGET_DIR = (
    PROJECT_ROOT
    / "data"
    / "gold"
    / "open_meteo"
    / "forecast"
)

print("Diretório do projeto:", PROJECT_ROOT)
print("Origem Silver:", SOURCE_DIR)
print("Destino Gold:", TARGET_DIR)

print("\nSilver existe?", SOURCE_DIR.exists())
print("Gold existe?", TARGET_DIR.exists())

Diretório do projeto: /home/mariarita-resende/Projetos/open_meteo_ingestion
Origem Silver: /home/mariarita-resende/Projetos/open_meteo_ingestion/data/silver/open_meteo/forecast
Destino Gold: /home/mariarita-resende/Projetos/open_meteo_ingestion/data/gold/open_meteo/forecast

Silver existe? True
Gold existe? True


In [13]:
def get_parquet_files(directory: Path) -> list[Path]:
    if not directory.exists():
        raise FileNotFoundError(
            f"Diretório Silver não encontrado: {directory}"
        )

    return sorted(directory.rglob("*.parquet"))


parquet_files = get_parquet_files(SOURCE_DIR)

print(f"Total de arquivos Parquet encontrados: {len(parquet_files)}")

print("\nPrimeiros 5 arquivos:")

for path in parquet_files[:5]:
    print(path)

Total de arquivos Parquet encontrados: 61

Primeiros 5 arquivos:
/home/mariarita-resende/Projetos/open_meteo_ingestion/data/silver/open_meteo/forecast/20260423/amsterdam_20260423.parquet
/home/mariarita-resende/Projetos/open_meteo_ingestion/data/silver/open_meteo/forecast/20260423/barcelona_20260423.parquet
/home/mariarita-resende/Projetos/open_meteo_ingestion/data/silver/open_meteo/forecast/20260423/berlin_20260423.parquet
/home/mariarita-resende/Projetos/open_meteo_ingestion/data/silver/open_meteo/forecast/20260423/brussels_20260423.parquet
/home/mariarita-resende/Projetos/open_meteo_ingestion/data/silver/open_meteo/forecast/20260423/budapest_20260423.parquet


In [14]:
sample_path = parquet_files[0]

sample_df = pd.read_parquet(
    sample_path,
    engine="pyarrow"
)

print("Arquivo analisado:", sample_path.name)
print("Quantidade de linhas:", len(sample_df))
print("Quantidade de colunas:", len(sample_df.columns))

print("\nColunas:")
print(sample_df.columns.tolist())

print("\nTipos de dados:")
print(sample_df.dtypes)

print("\nPrimeiras 5 linhas:")
display(sample_df.head())

Arquivo analisado: amsterdam_20260423.parquet
Quantidade de linhas: 168
Quantidade de colunas: 13

Colunas:
['source_name', 'source_type', 'city_name', 'ingestion_date', 'observation_time', 'temperature_celsius', 'feels_like_celsius', 'relative_humidity_pct', 'precipitation_probability', 'precipitation_mm', 'weather_code', 'wind_speed_10m', 'wind_direction_10m']

Tipos de dados:
source_name                   object
source_type                   object
city_name                     object
ingestion_date                object
observation_time              object
temperature_celsius          float64
feels_like_celsius           float64
relative_humidity_pct          int64
precipitation_probability      int64
precipitation_mm             float64
weather_code                   int64
wind_speed_10m               float64
wind_direction_10m             int64
dtype: object

Primeiras 5 linhas:


,source_name,source_type,city_name,ingestion_date,observation_time,temperature_celsius,feels_like_celsius,relative_humidity_pct,precipitation_probability,precipitation_mm,weather_code,wind_speed_10m,wind_direction_10m
0,open_meteo,forecast,amsterdam,20260423,2026-04-23T00:00Z,8.8,6.1,80,0,0.0,0,10.8,20
1,open_meteo,forecast,amsterdam,20260423,2026-04-23T01:00Z,8.7,6.1,78,0,0.0,0,9.4,36
2,open_meteo,forecast,amsterdam,20260423,2026-04-23T02:00Z,8.4,5.6,81,0,0.0,0,11.2,33
3,open_meteo,forecast,amsterdam,20260423,2026-04-23T03:00Z,8.0,5.3,82,0,0.0,0,10.4,28
4,open_meteo,forecast,amsterdam,20260423,2026-04-23T04:00Z,8.0,5.4,82,0,0.0,0,9.4,14


In [15]:
print("Valores ausentes por coluna:")
print(sample_df.isna().sum())

print("\nTotal de valores ausentes:")
print(sample_df.isna().sum().sum())

print("\nPercentual de valores ausentes por coluna:")
print((sample_df.isna().mean() * 100).round(2))

Valores ausentes por coluna:
source_name                  0
source_type                  0
city_name                    0
ingestion_date               0
observation_time             0
temperature_celsius          0
feels_like_celsius           0
relative_humidity_pct        0
precipitation_probability    0
precipitation_mm             0
weather_code                 0
wind_speed_10m               0
wind_direction_10m           0
dtype: int64

Total de valores ausentes:
0

Percentual de valores ausentes por coluna:
source_name                  0.0
source_type                  0.0
city_name                    0.0
ingestion_date               0.0
observation_time             0.0
temperature_celsius          0.0
feels_like_celsius           0.0
relative_humidity_pct        0.0
precipitation_probability    0.0
precipitation_mm             0.0
weather_code                 0.0
wind_speed_10m               0.0
wind_direction_10m           0.0
dtype: float64


In [32]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "pipelines":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Raiz do projeto: {PROJECT_ROOT}")

Raiz do projeto: /home/mariarita-resende/Projetos/open_meteo_ingestion


In [33]:
from transformations.forecast import build_daily_forecast_dataframe

In [17]:
daily_sample_df = build_daily_forecast_dataframe(sample_df)

print("Arquivo de origem:", sample_path.name)
print("Linhas horárias:", len(sample_df))
print("Linhas diárias:", len(daily_sample_df))
print("Quantidade de colunas Gold:", len(daily_sample_df.columns))

print("\nColunas:")
print(daily_sample_df.columns.tolist())

print("\nResultado da agregação diária:")
display(daily_sample_df)

Arquivo de origem: amsterdam_20260423.parquet
Linhas horárias: 168
Linhas diárias: 7
Quantidade de colunas Gold: 14

Colunas:
['source_name', 'source_type', 'city_name', 'ingestion_date', 'observation_date', 'temperature_mean_celsius', 'temperature_min_celsius', 'temperature_max_celsius', 'feels_like_mean_celsius', 'relative_humidity_mean_pct', 'precipitation_probability_mean_pct', 'precipitation_probability_max_pct', 'precipitation_total_mm', 'wind_speed_mean_kmh']

Resultado da agregação diária:


,source_name,source_type,city_name,ingestion_date,observation_date,temperature_mean_celsius,temperature_min_celsius,temperature_max_celsius,feels_like_mean_celsius,relative_humidity_mean_pct,precipitation_probability_mean_pct,precipitation_probability_max_pct,precipitation_total_mm,wind_speed_mean_kmh
0,open_meteo,forecast,amsterdam,20260423,2026-04-23,11.387500,7.6,16.8,8.629167,63.416667,0.000000,0,0.0,11.112500
1,open_meteo,forecast,amsterdam,20260423,2026-04-24,11.062500,7.3,16.4,8.083333,67.083333,0.000000,0,0.0,13.445833
2,open_meteo,forecast,amsterdam,20260423,2026-04-25,10.862500,7.4,15.6,8.062500,64.666667,0.000000,0,0.0,11.229167
3,open_meteo,forecast,amsterdam,20260423,2026-04-26,8.362500,4.2,12.9,6.066667,73.958333,0.000000,0,0.0,6.941667
4,open_meteo,forecast,amsterdam,20260423,2026-04-27,8.737500,3.8,13.5,6.979167,78.083333,2.583333,8,2.7,4.695833
5,open_meteo,forecast,amsterdam,20260423,2026-04-28,8.620833,6.4,10.9,5.629167,80.666667,5.041667,12,1.2,12.700000
6,open_meteo,forecast,amsterdam,20260423,2026-04-29,10.408333,5.9,14.1,6.583333,62.208333,0.500000,2,0.0,16.633333


In [18]:
processed_files = 0
total_daily_rows = 0
errors = []

for path in parquet_files:
    try:
        # Ler o arquivo Parquet da camada Silver
        df = pd.read_parquet(path, engine="pyarrow")

        # Calcular as métricas diárias
        daily_df = build_daily_forecast_dataframe(df)

        # Preservar a estrutura de diretórios da origem
        relative_path = path.relative_to(SOURCE_DIR)
        output_path = TARGET_DIR / relative_path

        # Criar o diretório de destino, se necessário
        output_path.parent.mkdir(parents=True, exist_ok=True)

        # Salvar o resultado em Parquet
        daily_df.to_parquet(
            output_path,
            engine="pyarrow",
            index=False
        )

        processed_files += 1
        total_daily_rows += len(daily_df)

    except Exception as error:
        errors.append({
            "arquivo": str(path),
            "erro": str(error)
        })

print("Arquivos processados:", processed_files)
print("Total de linhas diárias geradas:", total_daily_rows)
print("Quantidade de erros:", len(errors))

if errors:
    print("\nDetalhes dos erros:")
    for error in errors:
        print(error)

Arquivos processados: 61
Total de linhas diárias geradas: 427
Quantidade de erros: 0


In [19]:
# Listar os arquivos Parquet gerados na camada Gold
gold_files = sorted(TARGET_DIR.rglob("*.parquet"))

print("Total de arquivos Parquet na Gold:", len(gold_files))

# Selecionar um arquivo para validação
gold_sample_path = gold_files[0]

gold_sample_df = pd.read_parquet(
    gold_sample_path,
    engine="pyarrow"
)

print("\nArquivo analisado:", gold_sample_path.name)
print("Quantidade de linhas:", len(gold_sample_df))
print("Quantidade de colunas:", len(gold_sample_df.columns))

print("\nValores ausentes por coluna:")
print(gold_sample_df.isna().sum())

print("\nTotal de valores ausentes:")
print(gold_sample_df.isna().sum().sum())

print("\nDados gerados:")
display(gold_sample_df)

Total de arquivos Parquet na Gold: 61

Arquivo analisado: amsterdam_20260423.parquet
Quantidade de linhas: 7
Quantidade de colunas: 14

Valores ausentes por coluna:
source_name                           0
source_type                           0
city_name                             0
ingestion_date                        0
observation_date                      0
temperature_mean_celsius              0
temperature_min_celsius               0
temperature_max_celsius               0
feels_like_mean_celsius               0
relative_humidity_mean_pct            0
precipitation_probability_mean_pct    0
precipitation_probability_max_pct     0
precipitation_total_mm                0
wind_speed_mean_kmh                   0
dtype: int64

Total de valores ausentes:
0

Dados gerados:


,source_name,source_type,city_name,ingestion_date,observation_date,temperature_mean_celsius,temperature_min_celsius,temperature_max_celsius,feels_like_mean_celsius,relative_humidity_mean_pct,precipitation_probability_mean_pct,precipitation_probability_max_pct,precipitation_total_mm,wind_speed_mean_kmh
0,open_meteo,forecast,amsterdam,20260423,2026-04-23,11.387500,7.6,16.8,8.629167,63.416667,0.000000,0,0.0,11.112500
1,open_meteo,forecast,amsterdam,20260423,2026-04-24,11.062500,7.3,16.4,8.083333,67.083333,0.000000,0,0.0,13.445833
2,open_meteo,forecast,amsterdam,20260423,2026-04-25,10.862500,7.4,15.6,8.062500,64.666667,0.000000,0,0.0,11.229167
3,open_meteo,forecast,amsterdam,20260423,2026-04-26,8.362500,4.2,12.9,6.066667,73.958333,0.000000,0,0.0,6.941667
4,open_meteo,forecast,amsterdam,20260423,2026-04-27,8.737500,3.8,13.5,6.979167,78.083333,2.583333,8,2.7,4.695833
5,open_meteo,forecast,amsterdam,20260423,2026-04-28,8.620833,6.4,10.9,5.629167,80.666667,5.041667,12,1.2,12.700000
6,open_meteo,forecast,amsterdam,20260423,2026-04-29,10.408333,5.9,14.1,6.583333,62.208333,0.500000,2,0.0,16.633333


In [20]:
## Validação da camada Gold
def validate_gold_directory(directory: Path) -> dict:
    files = sorted(directory.rglob("*.parquet"))

    total_rows = 0
    unreadable_files = []
    files_with_missing_values = []
    empty_files = []

    for path in files:
        try:
            df = pd.read_parquet(path, engine="pyarrow")

            total_rows += len(df)

            if df.empty:
                empty_files.append(str(path))

            if df.isna().any().any():
                files_with_missing_values.append(str(path))

        except Exception as error:
            unreadable_files.append({
                "arquivo": str(path),
                "erro": str(error),
            })

    return {
        "diretorio": str(directory),
        "total_arquivos": len(files),
        "total_linhas": total_rows,
        "arquivos_vazios": empty_files,
        "arquivos_com_valores_ausentes": files_with_missing_values,
        "arquivos_com_erro_de_leitura": unreadable_files,
    }


archive_result = validate_gold_directory(
    PROJECT_ROOT / "data" / "gold" / "open_meteo" / "archive"
)

forecast_result = validate_gold_directory(
    PROJECT_ROOT / "data" / "gold" / "open_meteo" / "forecast"
)

for name, result in [
    ("ARCHIVE", archive_result),
    ("FORECAST", forecast_result),
]:
    print(f"\n{'=' * 15} {name} {'=' * 15}")
    print("Arquivos:", result["total_arquivos"])
    print("Linhas:", result["total_linhas"])
    print("Arquivos vazios:", len(result["arquivos_vazios"]))
    print(
        "Arquivos com valores ausentes:",
        len(result["arquivos_com_valores_ausentes"]),
    )
    print(
        "Arquivos com erro de leitura:",
        len(result["arquivos_com_erro_de_leitura"]),
    )


=============== ARCHIVE ===============
Arquivos: 68
Linhas: 140
Arquivos vazios: 0
Arquivos com valores ausentes: 0
Arquivos com erro de leitura: 0

=============== FORECAST ===============
Arquivos: 61
Linhas: 427
Arquivos vazios: 0
Arquivos com valores ausentes: 0
Arquivos com erro de leitura: 0


In [21]:
## Validação da camada Gold
def check_gold_consistency(directory: Path, source_type: str) -> dict:
    files = sorted(directory.rglob("*.parquet"))

    invalid_temperature = []
    invalid_probability = []
    duplicate_dates = []

    for path in files:
        df = pd.read_parquet(path, engine="pyarrow")

        # Verificar se a temperatura mínima supera a máxima
        invalid_temp = df[
            df["temperature_min_celsius"]
            > df["temperature_max_celsius"]
        ]

        if not invalid_temp.empty:
            invalid_temperature.append(str(path))

        # Verificar a probabilidade de precipitação no Forecast
        if source_type == "forecast":
            invalid_prob = df[
                (df["precipitation_probability_mean_pct"] < 0)
                | (df["precipitation_probability_mean_pct"] > 100)
                | (df["precipitation_probability_max_pct"] < 0)
                | (df["precipitation_probability_max_pct"] > 100)
            ]

            if not invalid_prob.empty:
                invalid_probability.append(str(path))

        # Verificar datas diárias duplicadas por cidade e ingestão
        duplicated = df.duplicated(
            subset=[
                "source_name",
                "source_type",
                "city_name",
                "ingestion_date",
                "observation_date",
            ],
            keep=False,
        )

        if duplicated.any():
            duplicate_dates.append(str(path))

    return {
        "arquivos_analisados": len(files),
        "arquivos_com_temperatura_inconsistente": invalid_temperature,
        "arquivos_com_probabilidade_inconsistente": invalid_probability,
        "arquivos_com_datas_duplicadas": duplicate_dates,
    }


archive_checks = check_gold_consistency(
    PROJECT_ROOT / "data" / "gold" / "open_meteo" / "archive",
    "archive",
)

forecast_checks = check_gold_consistency(
    PROJECT_ROOT / "data" / "gold" / "open_meteo" / "forecast",
    "forecast",
)

for name, result in [
    ("ARCHIVE", archive_checks),
    ("FORECAST", forecast_checks),
]:
    print(f"\n{'=' * 15} {name} {'=' * 15}")
    print("Arquivos analisados:", result["arquivos_analisados"])
    print(
        "Arquivos com temperatura inconsistente:",
        len(result["arquivos_com_temperatura_inconsistente"]),
    )
    print(
        "Arquivos com probabilidade inconsistente:",
        len(result["arquivos_com_probabilidade_inconsistente"]),
    )
    print(
        "Arquivos com datas duplicadas:",
        len(result["arquivos_com_datas_duplicadas"]),
    )


=============== ARCHIVE ===============
Arquivos analisados: 68
Arquivos com temperatura inconsistente: 0
Arquivos com probabilidade inconsistente: 0
Arquivos com datas duplicadas: 0

=============== FORECAST ===============
Arquivos analisados: 61
Arquivos com temperatura inconsistente: 0
Arquivos com probabilidade inconsistente: 0
Arquivos com datas duplicadas: 0


In [22]:
# Localizar o arquivo Silver de Amsterdam coletado hoje
source_path = next(
    p for p in SOURCE_DIR.rglob("amsterdam_20261015.parquet")
)

# Ler o arquivo Silver
amsterdam_silver = pd.read_parquet(source_path)

# Transformar os dados horários em resumos diários
amsterdam_gold = build_daily_forecast_dataframe(amsterdam_silver)

# Conferir o resultado
print("Arquivo de origem:", source_path.name)
print("Linhas na Silver:", len(amsterdam_silver))
print("Linhas na Gold:", len(amsterdam_gold))
print("Colunas na Gold:", len(amsterdam_gold.columns))

print("\nResumo diário:")
display(amsterdam_gold)

Arquivo de origem: amsterdam_20261015.parquet
Linhas na Silver: 168
Linhas na Gold: 7
Colunas na Gold: 14

Resumo diário:


,source_name,source_type,city_name,ingestion_date,observation_date,temperature_mean_celsius,temperature_min_celsius,temperature_max_celsius,feels_like_mean_celsius,relative_humidity_mean_pct,precipitation_probability_mean_pct,precipitation_probability_max_pct,precipitation_total_mm,wind_speed_mean_kmh
0,open_meteo,forecast,amsterdam,20261015,2026-10-15,15.575000,11.9,19.2,14.308333,78.916667,0.000000,0,0.00,12.675000
1,open_meteo,forecast,amsterdam,20261015,2026-10-16,16.008333,12.3,18.8,14.733333,79.875000,31.291667,87,2.10,14.158333
2,open_meteo,forecast,amsterdam,20261015,2026-10-17,12.150000,10.3,14.7,8.758333,75.208333,29.416667,85,19.60,19.108333
3,open_meteo,forecast,amsterdam,20261015,2026-10-18,10.062500,6.9,12.7,8.004167,83.875000,17.333333,27,1.76,9.233333
4,open_meteo,forecast,amsterdam,20261015,2026-10-19,10.987500,8.8,13.4,8.491667,84.666667,7.583333,25,0.60,14.091667
5,open_meteo,forecast,amsterdam,20261015,2026-10-20,9.879167,7.1,13.3,7.925000,91.416667,0.791667,2,0.00,10.579167
6,open_meteo,forecast,amsterdam,20261015,2026-10-21,10.770833,7.7,12.8,8.887500,91.541667,9.791667,14,0.60,11.758333


In [23]:
# Criar o diretório Gold correspondente à data da coleta
output_dir = TARGET_DIR / source_path.parent.name
output_dir.mkdir(parents=True, exist_ok=True)

# Definir o caminho do arquivo
output_path = output_dir / f"{source_path.stem}.parquet"

# Salvar os dados agregados em Parquet
amsterdam_gold.to_parquet(
    output_path,
    engine="pyarrow",
    index=False
)

print("Transformação Silver → Gold concluída!")
print("Arquivo salvo em:", output_path)
print("Tamanho:", output_path.stat().st_size, "bytes")

Transformação Silver → Gold concluída!
Arquivo salvo em: /home/mariarita-resende/Projetos/open_meteo_ingestion/data/gold/open_meteo/forecast/20261015/amsterdam_20261015.parquet
Tamanho: 10219 bytes


In [24]:
# Ler novamente o arquivo Gold
gold_check = pd.read_parquet(output_path, engine="pyarrow")

print("Arquivo lido:", output_path.name)
print("Quantidade de linhas:", len(gold_check))
print("Quantidade de colunas:", len(gold_check.columns))
print("Valores ausentes:", gold_check.isna().sum().sum())

print("\nDatas disponíveis:")
print(gold_check["observation_date"].tolist())

print("\nResumo diário:")
display(gold_check)

Arquivo lido: amsterdam_20261015.parquet
Quantidade de linhas: 7
Quantidade de colunas: 14
Valores ausentes: 0

Datas disponíveis:
['2026-10-15', '2026-10-16', '2026-10-17', '2026-10-18', '2026-10-19', '2026-10-20', '2026-10-21']

Resumo diário:


,source_name,source_type,city_name,ingestion_date,observation_date,temperature_mean_celsius,temperature_min_celsius,temperature_max_celsius,feels_like_mean_celsius,relative_humidity_mean_pct,precipitation_probability_mean_pct,precipitation_probability_max_pct,precipitation_total_mm,wind_speed_mean_kmh
0,open_meteo,forecast,amsterdam,20261015,2026-10-15,15.575000,11.9,19.2,14.308333,78.916667,0.000000,0,0.00,12.675000
1,open_meteo,forecast,amsterdam,20261015,2026-10-16,16.008333,12.3,18.8,14.733333,79.875000,31.291667,87,2.10,14.158333
2,open_meteo,forecast,amsterdam,20261015,2026-10-17,12.150000,10.3,14.7,8.758333,75.208333,29.416667,85,19.60,19.108333
3,open_meteo,forecast,amsterdam,20261015,2026-10-18,10.062500,6.9,12.7,8.004167,83.875000,17.333333,27,1.76,9.233333
4,open_meteo,forecast,amsterdam,20261015,2026-10-19,10.987500,8.8,13.4,8.491667,84.666667,7.583333,25,0.60,14.091667
5,open_meteo,forecast,amsterdam,20261015,2026-10-20,9.879167,7.1,13.3,7.925000,91.416667,0.791667,2,0.00,10.579167
6,open_meteo,forecast,amsterdam,20261015,2026-10-21,10.770833,7.7,12.8,8.887500,91.541667,9.791667,14,0.60,11.758333


In [26]:
target_date = "20261015"

source_dir = SOURCE_DIR / target_date
silver_files = sorted(source_dir.glob("*.parquet"))

print("Diretório Silver:", source_dir)
print("Arquivos encontrados:", len(silver_files))

for path in silver_files:
    print(path.name)

Diretório Silver: /home/mariarita-resende/Projetos/open_meteo_ingestion/data/silver/open_meteo/forecast/20261015
Arquivos encontrados: 14
amsterdam_20261015.parquet
barcelona_20261015.parquet
berlin_20261015.parquet
brussels_20261015.parquet
budapest_20261015.parquet
copenhagen_20261015.parquet
dublin_20261015.parquet
lisbon_20261015.parquet
paris_20261015.parquet
prague_20261015.parquet
rome_20261015.parquet
sao_tiago_20261015.parquet
veghel_20261015.parquet
vienna_20261015.parquet


In [27]:
# Processar somente os arquivos da coleta de 15/10/2026

processed_files = 0
processed_rows = 0
processing_errors = []

output_dir = TARGET_DIR / target_date
output_dir.mkdir(parents=True, exist_ok=True)

for path in silver_files:
    try:
        # Ler os dados da camada Silver
        df = pd.read_parquet(path, engine="pyarrow")

        # Transformar dados horários em dados diários
        daily_df = build_daily_forecast_dataframe(df)

        # Definir o arquivo de destino na Gold
        output_path = output_dir / path.name

        # Salvar o resultado
        daily_df.to_parquet(
            output_path,
            engine="pyarrow",
            index=False
        )

        processed_files += 1
        processed_rows += len(daily_df)

        print(
            f"OK: {path.name} -> "
            f"{len(daily_df)} linhas diárias"
        )

    except Exception as e:
        processing_errors.append((path.name, str(e)))
        print(f"ERRO: {path.name}: {e}")

print("\n========== RESUMO GOLD ==========")
print("Arquivos selecionados:", len(silver_files))
print("Arquivos processados:", processed_files)
print("Total de linhas diárias:", processed_rows)
print("Arquivos com erro:", len(processing_errors))

OK: amsterdam_20261015.parquet -> 7 linhas diárias
OK: barcelona_20261015.parquet -> 7 linhas diárias
OK: berlin_20261015.parquet -> 7 linhas diárias
OK: brussels_20261015.parquet -> 7 linhas diárias
OK: budapest_20261015.parquet -> 7 linhas diárias
OK: copenhagen_20261015.parquet -> 7 linhas diárias
OK: dublin_20261015.parquet -> 7 linhas diárias
OK: lisbon_20261015.parquet -> 7 linhas diárias
OK: paris_20261015.parquet -> 7 linhas diárias
OK: prague_20261015.parquet -> 7 linhas diárias
OK: rome_20261015.parquet -> 7 linhas diárias
OK: sao_tiago_20261015.parquet -> 7 linhas diárias
OK: veghel_20261015.parquet -> 7 linhas diárias
OK: vienna_20261015.parquet -> 7 linhas diárias

========== RESUMO GOLD ==========
Arquivos selecionados: 14
Arquivos processados: 14
Total de linhas diárias: 98
Arquivos com erro: 0


In [28]:
import pandas as pd

# Diretório Gold da coleta atual
gold_dir = TARGET_DIR / "20261015"
gold_files = sorted(gold_dir.glob("*.parquet"))

total_rows = 0
total_nulls = 0
total_columns = set()
errors = []

print("========== VALIDAÇÃO GOLD ==========")
print("Diretório:", gold_dir)
print("Arquivos encontrados:", len(gold_files))
print()

for path in gold_files:
    try:
        df = pd.read_parquet(path, engine="pyarrow")

        rows = len(df)
        columns = len(df.columns)
        nulls = int(df.isna().sum().sum())
        duplicated_dates = int(
            df.duplicated(
                subset=[
                    "source_name",
                    "source_type",
                    "city_name",
                    "ingestion_date",
                    "observation_date",
                ]
            ).sum()
        )

        total_rows += rows
        total_nulls += nulls
        total_columns.add(columns)

        status = (
            "OK"
            if rows == 7
            and columns == 14
            and nulls == 0
            and duplicated_dates == 0
            else "VERIFICAR"
        )

        print(
            f"{path.name}: "
            f"{rows} linhas | "
            f"{columns} colunas | "
            f"{nulls} nulos | "
            f"{duplicated_dates} duplicatas | "
            f"{status}"
        )

    except Exception as e:
        errors.append((path.name, str(e)))
        print(f"ERRO: {path.name}: {e}")

print("\n========== RESUMO FINAL GOLD ==========")
print("Arquivos encontrados:", len(gold_files))
print("Total de linhas:", total_rows)
print("Valores nulos:", total_nulls)
print("Quantidades de colunas encontradas:", total_columns)
print("Erros de leitura:", len(errors))

========== VALIDAÇÃO GOLD ==========
Diretório: /home/mariarita-resende/Projetos/open_meteo_ingestion/data/gold/open_meteo/forecast/20261015
Arquivos encontrados: 14

amsterdam_20261015.parquet: 7 linhas | 14 colunas | 0 nulos | 0 duplicatas | OK
barcelona_20261015.parquet: 7 linhas | 14 colunas | 0 nulos | 0 duplicatas | OK
berlin_20261015.parquet: 7 linhas | 14 colunas | 0 nulos | 0 duplicatas | OK
brussels_20261015.parquet: 7 linhas | 14 colunas | 0 nulos | 0 duplicatas | OK
budapest_20261015.parquet: 7 linhas | 14 colunas | 0 nulos | 0 duplicatas | OK
copenhagen_20261015.parquet: 7 linhas | 14 colunas | 0 nulos | 0 duplicatas | OK
dublin_20261015.parquet: 7 linhas | 14 colunas | 0 nulos | 0 duplicatas | OK
lisbon_20261015.parquet: 7 linhas | 14 colunas | 0 nulos | 0 duplicatas | OK
paris_20261015.parquet: 7 linhas | 14 colunas | 0 nulos | 0 duplicatas | OK
prague_20261015.parquet: 7 linhas | 14 colunas | 0 nulos | 0 duplicatas | OK
rome_20261015.parquet: 7 linhas | 14 colunas | 0 n